[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch08-model-training/04-di-chuyen-du-lieu.ipynb)

# Di chuyển dữ liệu: quy trình nạp, nạp trước, FlashAttention và AllReduce

Ba sổ tay trước đếm phép toán và byte nằm yên trong bộ nhớ. Sổ tay này đếm byte **đang di chuyển**:
từ đĩa lên CPU, từ CPU sang bộ tăng tốc, giữa bộ nhớ băng thông cao và SRAM trong attention, và giữa
các GPU khi đồng bộ gradient. Ở cả bốn chỗ, chương 8 đưa ra cùng một bài học: thông lượng (throughput)
của cả hệ là thông lượng của chặng chậm nhất, và cách sửa thường là di chuyển ít hơn hoặc chồng lấp
việc di chuyển với tính toán, chứ không phải tính nhanh hơn.

**Bạn sẽ làm:**
1. áp quy tắc chặng chậm nhất cho quy trình dữ liệu (data pipeline), rồi lần theo quy trình GPT-2 của
   chương từ đĩa tới PCIe;
2. đo trên CPU này xem nạp trước (prefetching) bằng một luồng nền tiết kiệm được gì, và vì sao GIL
   chặn luồng Python làm việc nặng về CPU;
3. viết attention chia khối (tiling) với softmax trực tuyến (online softmax), kiểm rằng nó cho đúng kết
   quả của attention chuẩn, rồi đếm số byte nó không phải ghi ra;
4. mô phỏng AllReduce vòng, đếm byte mỗi GPU gửi, rồi tính tường mạng của chương.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Chặng chậm nhất quyết định cả hệ

Chương viết thông lượng hệ thống là $R_{hệ} = \min(R_{quy trình}, R_{tính toán})$, và mức sử dụng
(utilization) bộ tăng tốc là $\min(1, R_{quy trình}/R_{tính toán})$. Dữ liệu còn đi qua ba tầng băng
thông, và tốc độ hiệu dụng là của tầng chậm nhất.

**Dự đoán:** một bộ tăng tốc xử lý được 1,000 ảnh mỗi giây được cấp 200 ảnh mỗi giây. Đổi sang bộ tăng
tốc gấp đôi thì mức sử dụng tăng hay giảm?

In [ ]:
def muc_su_dung(r_quy_trinh, r_tinh_toan):
    return min(1.0, r_quy_trinh / r_tinh_toan)


R_TINH_TOAN = sach(1000, nguon="1.000 ảnh/giây · 200 ảnh/giây → 20 phần trăm")
R_QUY_TRINH = sach(200, nguon="1.000 ảnh/giây · 200 ảnh/giây → 20 phần trăm")
theo_sach("mức sử dụng (%)", 100 * muc_su_dung(R_QUY_TRINH, R_TINH_TOAN), sach=20, nguon="200 ảnh/giây → 20 phần trăm")
theo_sach("bộ tăng tốc gấp đôi (%)", 100 * muc_su_dung(R_QUY_TRINH, sach(2000, nguon="2.000 ảnh/giây → 10 phần trăm")),
          sach=10, nguon="2.000 ảnh/giây → 10 phần trăm")
print(f"thông lượng hệ thống trong cả hai trường hợp: {min(R_QUY_TRINH, R_TINH_TOAN):,} ảnh mỗi giây")

tang = {"NVMe": sach(7, nguon="7 GB/s · 50 GB/s · 900 GB/s"),
        "RAM hệ thống": sach(50, nguon="7 GB/s · 50 GB/s · 900 GB/s"),
        "HBM của V100": sach(900, nguon="7 GB/s · 50 GB/s · 900 GB/s")}
cham_nhat = min(tang, key=tang.get)
print("\nba tầng: " + " · ".join(f"{ten} {bw} GB/s" for ten, bw in tang.items())
      + f" → tốc độ hiệu dụng {tang[cham_nhat]} GB/s, chặn bởi {cham_nhat}")
theo_sach("NVMe chỉ đạt một nửa đỉnh (GB/s)", tang["NVMe"] / 2, sach=3.50, nguon="3,50 GB/s")

T_TIEN_XU_LY = sach(40, nguon="40 ms · 10 ms → 4 worker")
T_TINH = sach(10, nguon="40 ms · 10 ms → 4 worker")
theo_sach("số worker tối thiểu", math.ceil(T_TIEN_XU_LY / T_TINH), sach=4, nguon="40 ms · 10 ms → 4 worker")
tang_cuong = [sach(10, nguon="10 + 15 + 5 = 30 ms"), sach(15, nguon="10 + 15 + 5 = 30 ms"), sach(5, nguon="10 + 15 + 5 = 30 ms")]
theo_sach("cắt, đổi màu, chuẩn hoá (ms)", sum(tang_cuong), sach=30, nguon="10 + 15 + 5 = 30 ms")
print()

Mức sử dụng **giảm** còn 10 phần trăm: bộ tăng tốc nhanh hơn chỉ chờ lâu hơn, còn thông lượng của hệ
đứng yên ở 200 ảnh mỗi giây. Trong ba tầng băng thông, đĩa chậm hơn HBM hơn hai bậc độ lớn.

### Quy trình dữ liệu GPT-2, đo tới cùng (Ví dụ 1.1)

Kịch bản của chương: một cụm 32 V100 huấn luyện GPT-2, với batch 32 chuỗi, mỗi chuỗi 1,024 token. Ba
chặng được tính: đọc kho WebText từ NVMe, tách token (tokenization) BPE trên một lõi CPU, và truyền
tensor qua PCIe.

**Dự đoán:** trong ba chặng ấy, chặng nào ăn gần hết ngân sách một bước?

In [ ]:
NVME = sach(7, nguon="40 GB · 7 GB/s · hệ số 0,1 → 0,70 GB/s")
HE_SO_TRUY_CAP = sach(0.1, nguon="hệ số 0,1 → 0,70 GB/s")
theo_sach("băng thông truy cập ngẫu nhiên (GB/s)", NVME * HE_SO_TRUY_CAP, sach=0.70, nguon="hệ số 0,1 → 0,70 GB/s")
T_DOC = sach(0.1, nguon="≈0,1 ms", tol=0.05)

B = sach(32, nguon="32×1.024 int64 · 262,1 KB")
S = sach(1024, nguon="32×1.024 int64 · 262,1 KB")
TOKEN_MOI_GIAY = sach(500_000, nguon="500 nghìn token/s")
theo_sach("token mỗi batch", B * S, sach=32_800, nguon="32,8 nghìn token")
t_tach = 1e3 * B * S / TOKEN_MOI_GIAY
theo_sach("tách token trên một lõi (ms)", t_tach, sach=65.5, nguon="≈65,5 ms", tol=0.05)

lo = torch.zeros(B, S, dtype=torch.int64)       # tensor sau khi đệm và đóng gói
so_byte = lo.numel() * lo.element_size()
theo_sach("tensor int64 (KB)", so_byte / 1e3, sach=262.1, nguon="32×1.024 int64 · 262,1 KB")
PCIE = sach(15.75, nguon="15,75 GB/s · 0,017 ms")
t_truyen = 1e3 * so_byte / (PCIE * 1e9)
theo_sach("truyền qua PCIe (ms)", t_truyen, sach=0.017, nguon="15,75 GB/s · 0,017 ms")

t_chuan_bi = T_DOC + t_tach + t_truyen
theo_sach("chuẩn bị trên một lõi (ms)", t_chuan_bi, sach=65.6, nguon="≈65,6 ms · ≈84,4 ms", tol=0.1)
T_BUOC = sach(84.4, nguon="≈84,4 ms", tol=0.05)
theo_sach("η khi chuẩn bị và tính chạy nối tiếp (%)", 100 * T_BUOC / (T_BUOC + t_chuan_bi), sach=56,
          nguon="≈84,4 ms · ≈56 phần trăm", tol=0.5)
for ten, t in (("đọc", T_DOC), ("tách token", t_tach), ("truyền PCIe", t_truyen)):
    print(f"   {ten:<12} {t:8.3f} ms  ({100 * t / t_chuan_bi:5.1f} % của phần chuẩn bị)")

Chặng ràng buộc không phải đĩa, không phải bus, mà là một lõi CPU tách token. Chương sửa nó bằng hai can
thiệp không đổi mô hình và không bớt một FLOP nào: 8 tiến trình công nhân (worker process), và nạp
trước để phần chuẩn bị chồng lên phần tính.

In [ ]:
N_WORKER = sach(8, nguon="8 worker")
t_tach_8 = t_tach / N_WORKER
theo_sach("tách token với 8 worker (ms)", t_tach_8, sach=8.2, nguon="8 worker · ≈8,2 ms", tol=0.05)
t_vong = max(T_BUOC, T_DOC + t_tach_8 + t_truyen)      # chồng lấp: một vòng là chặng chậm nhất
print(f"một vòng khi chồng lấp: max({T_BUOC}, {T_DOC + t_tach_8 + t_truyen:.1f}) = {t_vong} ms")
theo_sach("B / T_bước (mẫu mỗi giây)", B / (T_BUOC / 1e3), sach=379, nguon="379 mẫu/giây")
ETA_SAU = sach(95, nguon="95 phần trăm · 379 mẫu/giây")
print(f"ở mức sử dụng {ETA_SAU} %: {ETA_SAU / 100 * B / (T_BUOC / 1e3):.0f} mẫu mỗi giây")

379 mẫu mỗi giây là đúng $B / T_{bước}$, tức tốc độ bộ tăng tốc tiêu thụ khi nó **không bao giờ** phải
chờ. Chương ghi đồng thời mức sử dụng 95 phần trăm, mà ở mức ấy phép tính cho khoảng 360. Sổ tay giữ cả
hai: 379 là trần của kịch bản, 95 phần trăm là phần chương cho rằng nạp trước giành lại được.

## 2. Nạp trước, đo trên máy này

Nạp trước biến thời gian một vòng từ **tổng** các chặng thành **max** của chúng. Ô dưới mô phỏng điều
đó trên CPU. "Nạp" một batch là chờ 12 ms, như chờ đĩa hay mạng (`time.sleep` nhả GIL, đúng như
một thao tác vào/ra). "Tính" là sáu phép nhân ma trận thật. Bản tuần tự nạp rồi tính. Bản nạp trước
chạy việc nạp trên một luồng nền và đẩy batch vào một hàng đợi sâu 2, nên trong lúc batch này được tính,
batch sau đã đang nạp.

**Dự đoán:** với 20 batch, bản nạp trước tiết kiệm được gần bằng tổng thời gian nạp, hay ít hơn?

In [ ]:
import queue
import threading

dat_hat_giong()
W = torch.randn(512, 512)
x_mau = torch.randn(256, 512)
NAP_GIAY = 0.012


def nap():
    time.sleep(NAP_GIAY)
    return x_mau


def tinh(x):
    for _ in range(6):
        x = torch.tanh(x @ W)
    return x


def tuan_tu(n):
    nhat_ky, t0 = [], time.perf_counter()
    for _ in range(n):
        a = time.perf_counter()
        b = nap()
        c = time.perf_counter()
        tinh(b)
        d = time.perf_counter()
        nhat_ky += [("nạp", a - t0, c - t0), ("tính", c - t0, d - t0)]
    return time.perf_counter() - t0, nhat_ky


def nap_truoc(n, do_sau=2):
    hang_doi, nhat_ky, t0 = queue.Queue(maxsize=do_sau), [], time.perf_counter()

    def san_xuat():
        for _ in range(n):
            a = time.perf_counter()
            b = nap()
            nhat_ky.append(("nạp", a - t0, time.perf_counter() - t0))
            hang_doi.put(b)

    luong = threading.Thread(target=san_xuat)
    luong.start()
    for _ in range(n):
        b = hang_doi.get()
        c = time.perf_counter()
        tinh(b)
        nhat_ky.append(("tính", c - t0, time.perf_counter() - t0))
    luong.join()
    return time.perf_counter() - t0, nhat_ky


lan_tinh = []
for _ in range(6):
    t0 = time.perf_counter()
    tinh(x_mau)
    lan_tinh.append(time.perf_counter() - t0)
t_tinh = min(lan_tinh)
N_BATCH = 20
ket = {}
for ten, ham in (("tuần tự", tuan_tu), ("nạp trước", nap_truoc)):
    lan = [ham(N_BATCH) for _ in range(3)]
    ket[ten] = min(lan, key=lambda r: r[0])
    do_tren_may(f"{ten}, {N_BATCH} batch", 1e3 * ket[ten][0], "ms")
do_tren_may("một lần tính", 1e3 * t_tinh, "ms")
do_tren_may("tiết kiệm nhờ nạp trước", 100 * (1 - ket["nạp trước"][0] / ket["tuần tự"][0]), "%")

In [ ]:
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(9, 2.8))
for hang, ten in enumerate(("tuần tự", "nạp trước")):
    for loai, a, b in ket[ten][1]:
        if a > 0.25:
            continue
        y = hang * 1.0 + (0.18 if loai == "nạp" else -0.18)
        ax.barh(y, 1e3 * (b - a), left=1e3 * a, height=0.3, color="C1" if loai == "nạp" else "C0")
ax.set_yticks([0, 1], ["tuần tự", "nạp trước"])
ax.set_ylim(-0.5, 1.5)
ax.set_xlim(0, 250)
ax.set_xlabel("thời gian (ms, 250 ms đầu, đo trên máy này)")
ax.set_title("Nạp trước chồng việc nạp lên việc tính")
ax.legend(handles=[Patch(color="C1", label="nạp (chờ vào/ra)"), Patch(color="C0", label="tính")],
          loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Ở bản tuần tự, mỗi vòng là nạp cộng tính. Ở bản nạp trước, sau batch đầu tiên, việc nạp nằm trọn dưới
việc tính, nên một vòng chỉ còn là việc tính, chặng chậm hơn trong hai chặng. Nếu nạp lâu hơn tính thì
ngược lại: vòng là thời gian nạp, và bộ tăng tốc vẫn chờ, chỉ chờ ít hơn.

Chương có hai kịch bản minh hoạ cho đúng hiệu ứng này, với thời lượng giả định của chúng:

In [ ]:
truoc, sau = sach(95, nguon="≈95 phút → ≈65 phút", tol=0.5), sach(65, nguon="≈95 phút → ≈65 phút", tol=0.5)
theo_sach("chồng lấp, hình 10 và 11 (%)", 100 * (1 - sau / truoc), sach=31.6, nguon="31,6 phần trăm")
truoc, sau = sach(105, nguon="47,6 phần trăm · 105 → 55 phút"), sach(55, nguon="47,6 phần trăm · 105 → 55 phút")
theo_sach("ngộ nhận 'bộ tăng tốc luôn là điểm nghẽn' (%)", 100 * (1 - sau / truoc), sach=47.6, nguon="47,6 phần trăm")
print()

### Vì sao luồng Python không cứu được việc nặng về CPU

Nạp trước ở trên chạy được trên một luồng vì `time.sleep` nhả khoá GIL. Việc tiền xử lý viết bằng Python
thuần thì giữ GIL, nên nhiều luồng chỉ thay nhau chạy. Ô dưới chia 8 phần việc cho 1, 2 rồi 4 luồng, một
lần với vòng lặp Python thuần, một lần với việc chờ vào/ra.

**Dự đoán:** với vòng lặp Python thuần, 4 luồng nhanh hơn 1 luồng bao nhiêu lần?

In [ ]:
def viec_python(n=200_000):
    s = 0
    for i in range(n):
        s += i * i
    return s


def viec_vao_ra():
    time.sleep(0.01)


def chay_luong(viec, so_luong, tong=8):
    t0 = time.perf_counter()
    cac = [threading.Thread(target=lambda: [viec() for _ in range(tong // so_luong)]) for _ in range(so_luong)]
    for th in cac:
        th.start()
    for th in cac:
        th.join()
    return time.perf_counter() - t0


for ten, viec in (("Python thuần", viec_python), ("chờ vào/ra", viec_vao_ra)):
    mot = min(chay_luong(viec, 1) for _ in range(3))
    for k in (2, 4):
        do_tren_may(f"{ten}: {k} luồng nhanh hơn 1 luồng", mot / min(chay_luong(viec, k) for _ in range(3)), "lần")

T_CHUAN_BI_GIL = sach(250, nguon="250 ms · 12 ms")
T_BUOC_GIL = sach(12, nguon="250 ms · 12 ms")
theo_sach("η của 'GPU bị GIL khoá' (%)", 100 * T_BUOC_GIL / (T_BUOC_GIL + T_CHUAN_BI_GIL), sach=4.6,
          nguon="≈4,6 phần trăm", tol=0.05)
print()

Vòng lặp Python thuần không nhanh lên chút nào, vì GIL chỉ cho một luồng chạy mã Python tại một thời
điểm. Việc chờ vào/ra thì nhanh lên gần đúng theo số luồng. Đó là lý do chương nói worker phải là
**tiến trình** (`num_workers` lớn hơn 0 trong `DataLoader`), không phải luồng. Sổ tay này không bật
tiến trình công nhân, để chạy được ở mọi môi trường, nhưng phép tính của chương thì rõ: 250 ms chuẩn bị bị
GIL khoá trước một bước GPU 12 ms để lại mức sử dụng khoảng 4.6 phần trăm.

### Cái giá của nạp trước: bộ nhớ máy chủ

Mỗi batch trong bộ đệm nạp trước nằm trong RAM của máy chủ. Ví dụ của chương: batch 256 ảnh 1024×1024,
`num_workers = 4`, `prefetch_factor = 2`. Sổ tay giả định ảnh 3 kênh, mỗi giá trị FP32.

In [ ]:
N_ANH = sach(256, nguon="256 ảnh 1024×1024")
CANH = sach(1024, nguon="256 ảnh 1024×1024")
KENH = 3                                                     # giả định: ảnh RGB
mot_batch = N_ANH * CANH * CANH * KENH * torch.empty(0).element_size() / 1e9
theo_sach("một batch (GB)", mot_batch, sach=3.2, nguon="256 ảnh 1024×1024 → ≈3,2 GB", tol=0.05)
so_batch = sach(4, nguon="4 worker · prefetch 2 → 8 batch") * sach(2, nguon="4 worker · prefetch 2 → 8 batch")
theo_sach("số batch trong cửa sổ", so_batch, sach=8, nguon="prefetch 2 → 8 batch")
theo_sach("cửa sổ nạp trước (GB)", so_batch * mot_batch, sach=25.8, nguon="8 batch ≈25,8 GB", tol=0.05)
print()

## 3. FlashAttention: cùng kết quả, ít byte hơn

Attention chuẩn ghi trọn ma trận điểm $S \times S$ ra bộ nhớ băng thông cao rồi đọc lại nó. Chương đếm
mật độ tính toán (arithmetic intensity) của riêng phần ấy: phép $QK^\top$ của một đầu tốn khoảng
$2S^2 d_{đầu}$ FLOP, còn ghi rồi đọc lại ma trận điểm FP16 di chuyển khoảng $4S^2$ byte, nên mật độ là
$d_{đầu}/2$. Số FLOP thì không cần tin sách: `FlopCounterMode` đếm được.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

D_MO_HINH = sach(768, nguon="768 · 12 đầu · d_head=64 → 32 FLOP mỗi byte")
N_DAU = sach(12, nguon="768 · 12 đầu · d_head=64 → 32 FLOP mỗi byte")
D_DAU = D_MO_HINH // N_DAU
theo_sach("chiều mỗi đầu", D_DAU, sach=64, nguon="768 · 12 đầu · d_head=64")
S_DEM = 1024
q = torch.randn(S_DEM, D_DAU)
with FlopCounterMode(display=False) as dem_flop:
    q @ q.T
flop = dem_flop.get_total_flops()
assert flop == 2 * S_DEM**2 * D_DAU                               # đúng 2·S²·d_đầu
byte_diem = 2 * S_DEM**2 * torch.empty(0, dtype=torch.float16).element_size()   # ghi một lần, đọc một lần
print(f"S = {S_DEM:,}: {flop:,} FLOP, {byte_diem:,} byte qua lại của ma trận điểm FP16")
theo_sach("mật độ tính toán (FLOP mỗi byte)", flop / byte_diem, sach=32, nguon="d_head=64 → 32 FLOP mỗi byte")
print()

Tỉ số không phụ thuộc $S$: nó luôn là $d_{đầu}/2$. Ba mươi hai FLOP mỗi byte nằm dưới điểm gãy (ridge
point) của các bộ tăng tốc huấn luyện, nên phần này nghẽn ở băng thông. FlashAttention không giảm số
FLOP. Nó giảm số byte, bằng cách không bao giờ ghi ma trận điểm ra.

### Chia khối và softmax trực tuyến

Softmax thường cần cả hàng điểm trước khi tính được phần tử nào, vì mẫu số là tổng trên cả hàng.
Softmax trực tuyến đi qua các khối khoá, giá trị từng khối một và giữ hai thống kê cho mỗi hàng: giá trị
lớn nhất đang chạy $m$ và mẫu số đang chạy $\ell$. Mỗi khi khối mới tới với $m$ lớn hơn, phần đầu ra đã
cộng dồn được nhân lại với $e^{m_{cũ} - m_{mới}}$. Ô dưới viết đúng thuật toán ấy, với ô 128×128.

**Dự đoán:** kết quả có khớp attention chuẩn tới sai số làm tròn không, hay chỉ xấp xỉ?

In [ ]:
def attention_chuan(q, k, v):
    return torch.softmax(q @ k.T / math.sqrt(q.shape[1]), dim=-1) @ v


def attention_chia_khoi(q, k, v, Br=128, Bc=128):
    S_, d = q.shape
    ra = torch.empty_like(q)
    o_lon_nhat = 0
    for i in range(0, S_, Br):
        qi = q[i:i + Br]
        m = torch.full((qi.shape[0], 1), -math.inf, dtype=q.dtype)
        l = torch.zeros((qi.shape[0], 1), dtype=q.dtype)
        cong_don = torch.zeros((qi.shape[0], d), dtype=q.dtype)
        for j in range(0, S_, Bc):
            o = qi @ k[j:j + Bc].T / math.sqrt(d)                # một ô điểm, không bao giờ cả ma trận
            o_lon_nhat = max(o_lon_nhat, o.numel())
            m_moi = torch.maximum(m, o.max(dim=1, keepdim=True).values)
            p = torch.exp(o - m_moi)
            co_lai = torch.exp(m - m_moi)
            l = l * co_lai + p.sum(dim=1, keepdim=True)
            cong_don = cong_don * co_lai + p @ v[j:j + Bc]
            m = m_moi
        ra[i:i + Br] = cong_don / l
    return ra, o_lon_nhat


dat_hat_giong()
q, k, v = (torch.randn(1024, 64, dtype=torch.float64) for _ in range(3))
chuan = attention_chuan(q, k, v)
chia_khoi, o_lon_nhat = attention_chia_khoi(q, k, v)
sdpa = torch.nn.functional.scaled_dot_product_attention(q[None], k[None], v[None])[0]
lech, lech_sdpa = (chuan - chia_khoi).abs().max().item(), (chuan - sdpa).abs().max().item()
print(f"lệch lớn nhất so với attention chuẩn: chia khối {lech:.1e} · scaled_dot_product_attention {lech_sdpa:.1e}")
print(f"ô điểm lớn nhất từng tạo ra: {o_lon_nhat:,} phần tử, so với {1024 * 1024:,} của ma trận đầy đủ")
assert lech < 1e-12 and lech_sdpa < 1e-12 and o_lon_nhat == 128 * 128

Kết quả khớp tới sai số làm tròn của FP64: chia khối là một cách **sắp xếp lại** phép tính, không phải
một phép xấp xỉ. `torch.nn.functional.scaled_dot_product_attention` cho cùng kết quả. Đó là primitive mà
chương khuyên gọi thay cho attention viết tay, vì nó điều phối sang một kernel hợp nhất khi phần cứng và
đầu vào cho phép. Bản Python ở trên chạy chậm trên CPU, vì lợi ích của nó nằm ở lưu lượng HBM trên GPU.
Thứ đếm được ở đây là số byte:

In [ ]:
S_DAI = sach(4096, nguon="4.096 · 64 chiều → 67,1 MB")
D_DAU_DAI = sach(64, nguon="4.096 · 64 chiều → 67,1 MB")
B32 = torch.empty(0, dtype=torch.float32).element_size()
diem_day_du = S_DAI**2 * B32 / 1e6
theo_sach("ma trận điểm FP32 một đầu (MB)", diem_day_du, sach=67.1, nguon="4.096 · 64 chiều → 67,1 MB")
theo_sach("16 đầu (GB)", sach(16, nguon="16 đầu → 1,1 GB") * diem_day_du / 1e3, sach=1.1, nguon="16 đầu → 1,1 GB")
O = sach(128, nguon="128×128 → 65,5 KB")
theo_sach("một ô điểm FP32 (KB)", O * O * B32 / 1e3, sach=65.5, nguon="128×128 → 65,5 KB")
qkv = 3 * S_DAI * D_DAU_DAI * B32 / 1e6
theo_sach("FlashAttention giữ Q, K, V (MB mỗi đầu)", qkv, sach=3.1, nguon="≈3,1 MB mỗi đầu", tol=0.05)
theo_sach("mức giảm", diem_day_du / qkv, sach=21.3, nguon="21,3 lần")
print()

In [ ]:
cac_s = np.array([512, 1024, 2048, 4096, 8192, 16384])
fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(cac_s, cac_s**2 * B32 / 1e6, "o-", color="C3", label="attention chuẩn: ma trận điểm S×S")
ax.loglog(cac_s, 3 * cac_s * D_DAU_DAI * B32 / 1e6, "s-", color="C0", label="FlashAttention: Q, K, V (3·S·64)")
ax.axhline(O * O * B32 / 1e6, color="0.5", ls=":", lw=1)
ax.annotate("một ô 128×128 trong SRAM", (cac_s[0], O * O * B32 / 1e6), xytext=(0, 4), textcoords="offset points",
            fontsize=8, color="0.4")
ax.annotate(f"{diem_day_du:.1f} MB", (S_DAI, diem_day_du), xytext=(-48, 6), textcoords="offset points", fontsize=8,
            color="C3")
ax.annotate(f"{qkv:.1f} MB", (S_DAI, qkv), xytext=(4, -14), textcoords="offset points", fontsize=8, color="C0")
ax.set_xticks(cac_s, [f"{s:,}" for s in cac_s])
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_xlabel("độ dài chuỗi S")
ax.set_ylabel("MB mỗi đầu, FP32")
ax.set_title("Byte phải giữ: bậc hai so với bậc một (d_đầu = 64)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Hai đường gặp nhau quanh $S = 192$, tức $3 \cdot 64$. Ở chuỗi ngắn, chia khối không lợi gì về bộ nhớ,
đúng như chương nói lợi ích giảm dần ở chuỗi ngắn. Ở 4,096 token thì khoảng cách là 21.3 lần, và nó
tiếp tục gấp đôi mỗi khi $S$ gấp đôi.

Bảng benchmark của chương là **kịch bản minh hoạ** trên một cấu hình A100 40 GB, và nguyên lý rộng hơn
của nó là một phép tính về năng lượng: bỏ thêm 10 phần trăm số học để giảm nửa lưu lượng bộ nhớ thì
tiến tới mức giảm 2 lần, trong giới hạn mà di chuyển dữ liệu chiếm toàn bộ ngân sách.

In [ ]:
theo_sach("lượt xuôi ở 2,048 token", sach(45, nguon="45 → 15 ms · 3 lần") / sach(15, nguon="45 → 15 ms · 3 lần"),
          sach=3, nguon="45 → 15 ms · 3 lần")
theo_sach("lượt ngược ở 2,048 token", sach(120, nguon="120 → 35 ms · 3,4 lần") / sach(35, nguon="120 → 35 ms · 3,4 lần"),
          sach=3.4, nguon="120 → 35 ms · 3,4 lần")

THEM_SO_HOC = sach(10, nguon="10 phần trăm · 2 lần") / 100
phan_di_chuyen = np.linspace(0, 1, 101)                 # phần năng lượng dành cho di chuyển dữ liệu
loi = 1 / ((1 - phan_di_chuyen) * (1 + THEM_SO_HOC) + phan_di_chuyen / 2)
theo_sach("giới hạn khi di chuyển chiếm hết ngân sách", loi[-1], sach=2, nguon="10 phần trăm · 2 lần")
hoa_von = THEM_SO_HOC / (THEM_SO_HOC + 0.5)
print(f"đổi này chỉ có lợi khi di chuyển dữ liệu chiếm hơn {100 * hoa_von:.1f} % năng lượng;"
      f" ở 50 % thì lợi {1 / (0.5 * (1 + THEM_SO_HOC) + 0.25):.2f} lần")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(100 * phan_di_chuyen, loi, color="C0")
ax.axhline(1, color="0.5", lw=0.8)
ax.axvline(100 * hoa_von, color="0.5", ls=":", lw=1)
ax.annotate(f"hoà vốn ở {100 * hoa_von:.1f} %", (100 * hoa_von, 1), xytext=(6, 10), textcoords="offset points", fontsize=8)
ax.annotate("2 lần khi di chuyển là tất cả", (100, loi[-1]), xytext=(-150, -4), textcoords="offset points", fontsize=8)
ax.set_xlabel("phần năng lượng dành cho di chuyển dữ liệu (%)")
ax.set_ylabel("mức giảm năng lượng (lần)")
ax.set_title("Thêm 10 % số học, bớt một nửa lưu lượng")
plt.tight_layout()
plt.show()

Giảm nửa một số hạng của tổng không thể giảm tổng đi hơn một nửa: mức lợi thật luôn dưới 2 lần, và nó
thành lỗ khi số học chiếm phần lớn năng lượng.

## 4. AllReduce vòng và tường mạng

Với song song dữ liệu (data parallelism), mỗi GPU giữ một bản sao mô hình, tính gradient trên phần dữ
liệu của nó, rồi mọi GPU phải cộng gradient lại. AllReduce vòng làm việc đó trong hai pha trên một vòng
$N$ GPU. Mỗi gradient được chia thành $N$ mảnh. Pha một, sau $N - 1$ bước, mỗi GPU giữ tổng đầy đủ của
một mảnh. Pha hai, cũng $N - 1$ bước, chuyền các mảnh đã cộng đi hết vòng. Ô dưới mô phỏng đúng như vậy
bằng NumPy, và đếm byte mỗi GPU gửi.

**Dự đoán:** mỗi GPU gửi bao nhiêu lần kích thước gradient khi $N = 8$? Khi $N$ rất lớn?

In [ ]:
def allreduce_vong(cac_gradient):
    N = len(cac_gradient)
    manh = [np.array_split(g.copy(), N) for g in cac_gradient]
    da_gui = [0] * N
    for pha in ("cộng dồn", "chuyền đi"):
        for buoc in range(N - 1):
            thu = []
            for r in range(N):
                c = (r - buoc) % N if pha == "cộng dồn" else (r + 1 - buoc) % N
                thu.append(((r + 1) % N, c, manh[r][c].copy()))
                da_gui[r] += manh[r][c].nbytes
            for dich, c, du_lieu in thu:
                if pha == "cộng dồn":
                    manh[dich][c] += du_lieu
                else:
                    manh[dich][c] = du_lieu
    return [np.concatenate(m) for m in manh], da_gui


rng = np.random.default_rng(SEED)
he_so_gui = {}
for N in (2, 4, 8, 16, 32):
    cac_gradient = [rng.standard_normal(9600) for _ in range(N)]
    ket_qua, da_gui = allreduce_vong(cac_gradient)
    assert all(np.allclose(r, sum(cac_gradient)) for r in ket_qua)    # mọi GPU có đúng tổng
    assert len(set(da_gui)) == 1                                         # mọi GPU gửi như nhau
    he_so_gui[N] = da_gui[0] / cac_gradient[0].nbytes
    assert he_so_gui[N] == 2 * (N - 1) / N
    print(f"N = {N:>2}: mỗi GPU gửi {he_so_gui[N]:.4f} lần gradient, đúng 2(N − 1)/N")
N_GPU = sach(8, trich="scales training to 8 GPUs, one per node")
theo_sach("hệ số gửi, 8 GPU", he_so_gui[N_GPU], sach=1.75, nguon="14 GB · 1,75 lần")
print(f"khi N rất lớn, hệ số tiến về 2: {2 * (1000 - 1) / 1000} ở N = 1,000")

Mỗi GPU gửi gần hai bản gradient, và con số ấy gần như không đổi theo $N$: AllReduce vòng không làm số
byte mỗi GPU phình theo kích thước cụm. Nhưng hai bản gradient của một mô hình lớn vẫn là rất nhiều
byte. Napkin Math 1.10 của chương: mô hình 7 tỷ tham số, gradient FP16, 8 GPU, mỗi GPU một node, nối
bằng mạng 100 Gbps.

In [ ]:
GRAD = sach(7e9, nguon="7 tỷ · 14 GB FP16") * torch.empty(0, dtype=torch.float16).element_size() / 1e9
theo_sach("gradient FP16 (GB)", GRAD, sach=14, nguon="7 tỷ · 14 GB FP16")
gui = he_so_gui[N_GPU] * GRAD
theo_sach("byte mỗi GPU gửi (GB)", gui, sach=24.5, nguon="1,75 lần · 24,5 GB")
BW_MANG = sach(100, trich="connected by a 100 Gbps fabric") / 8          # Gbps → GB/s
theo_sach("băng thông mạng (GB/s)", BW_MANG, sach=12.5, nguon="24,5 GB · 12,5 GB/s")
t_dong_bo = gui / BW_MANG
theo_sach("thời gian đồng bộ, chặn dưới (giây)", t_dong_bo, sach=1.96, nguon="12,5 GB/s · 1,96 giây")
T_TINH_BUOC = sach(1, nguon="≈1 giây · 33,8 phần trăm", tol=0.5)
theo_sach("η_hw khi không chồng lấp (%)", 100 * T_TINH_BUOC / (T_TINH_BUOC + t_dong_bo), sach=33.8, nguon="33,8 phần trăm")
NVLINK = sach(900, nguon="900 GB/s so với 12,5 GB/s")
print(f"cùng lượng byte qua NVLink {NVLINK} GB/s: {gui / NVLINK:.3f} giây, nhanh hơn {NVLINK / BW_MANG:.0f} lần")

In [ ]:
cac_n = np.array([1, 2, 4, 8, 16, 32, 64, 128])
fig, ax = plt.subplots(figsize=(8, 4))
for ten, bw, mau in (("mạng 100 Gbps (12.5 GB/s)", BW_MANG, "C3"), ("NVLink (900 GB/s)", NVLINK, "C0")):
    t_db = np.where(cac_n > 1, 2 * (cac_n - 1) / cac_n * GRAD / bw, 0.0)
    ax.semilogx(cac_n, cac_n * T_TINH_BUOC / (T_TINH_BUOC + t_db), "o-", color=mau, label=ten)
ax.semilogx(cac_n, cac_n, "--", color="0.5", lw=1, label="lý tưởng")
eta_8 = T_TINH_BUOC / (T_TINH_BUOC + t_dong_bo)
ax.annotate(f"8 GPU: {N_GPU * eta_8:.1f} lần", (N_GPU, N_GPU * eta_8), xytext=(8, -14), textcoords="offset points",
            fontsize=8, color="C3")
ax.set_xticks(cac_n, [str(n) for n in cac_n])
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_yscale("log")
ax.set_yticks([1, 2, 4, 8, 16, 32, 64, 128], ["1", "2", "4", "8", "16", "32", "64", "128"])
ax.yaxis.set_minor_locator(plt.NullLocator())
ax.set_xlabel("số GPU (mỗi GPU một node, AllReduce vòng)")
ax.set_ylabel("mức tăng tốc so với 1 GPU")
ax.set_title("Tường mạng: 7 tỷ tham số, 1 giây tính mỗi bước (giả định)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Đường đỏ mở rộng giả định của Napkin Math 1.10 ra mọi $N$: mỗi bước vẫn một giây tính, và đồng bộ không
chồng lên tính. Ở 8 GPU, cụm chỉ nhanh hơn một GPU chưa tới 3 lần. Và vì hệ số gửi tiến về 2, phần đồng
bộ không giảm khi thêm GPU, nên mức tăng tốc chỉ tăng theo tỉ lệ $\eta$ cố định ấy. Qua NVLink, cùng
lượng byte gần như không đáng kể. Đó là lý do chương nói phân cấp băng thông quyết định chiến lược: giữ
trao đổi dày đặc trong node, và chỉ đưa ra mạng những gì buộc phải đưa.

Phần ngộ nhận của chương đưa một kịch bản giả định nhẹ hơn: đồng bộ chiếm 30 đến 50 phần trăm mỗi bước.

In [ ]:
ly_tuong = sach(8, nguon="4 đến 6 lần thay vì 8")
for phan, sach_lan in ((sach(50, nguon="30 đến 50 phần trăm") / 100, sach(4, nguon="4 đến 6 lần thay vì 8")),
                       (sach(30, nguon="30 đến 50 phần trăm") / 100, sach(6, nguon="4 đến 6 lần thay vì 8"))):
    print(f"đồng bộ chiếm {100 * phan:.0f} %: {ly_tuong} × {1 - phan:.1f} = {ly_tuong * (1 - phan):.1f} lần, sách ghi {sach_lan}")

Với 30 phần trăm, phép nhân cho 5.6 lần, mà sách làm tròn lên 6. Khoảng 4 đến 6 lần ấy là ước lượng thô
của một kịch bản giả định. Chương nói rõ quyết định phải dựa trên thời gian đầu cuối và hiệu suất mở
rộng (scaling efficiency) **đo được**.

## Thử thêm

1. Ở ô `overlap`, đổi `NAP_GIAY` thành 0.03 để việc nạp lâu hơn việc tính. Nạp trước còn tiết kiệm
   được bao nhiêu phần trăm, và thời gian một vòng bây giờ là chặng nào?
2. Ở ô `tiled`, đổi `Br=128, Bc=128` thành `Br=64, Bc=256`, và thử một kích thước không chia hết như
   `Bc=100`. Kết quả có còn khớp không? Ô điểm lớn nhất bây giờ là bao nhiêu phần tử?
3. Ở ô `wall`, giả sử gradient được nén còn một nửa trước khi gửi. Mức sử dụng của 8 GPU thành bao
   nhiêu? Cần nén bao nhiêu lần để đồng bộ bằng đúng một giây tính?

## Tóm lại

* Thông lượng của quy trình huấn luyện là thông lượng của chặng chậm nhất. Bộ tăng tốc nhanh hơn trước
  một quy trình chậm chỉ làm mức sử dụng giảm, và trong ví dụ GPT-2 của chương, chặng ràng buộc là một
  lõi CPU tách token chứ không phải đĩa hay PCIe.
* Nạp trước đổi thời gian một vòng từ tổng các chặng thành chặng lớn nhất, với cái giá là RAM của máy
  chủ cho mỗi batch trong bộ đệm. Việc nặng về CPU viết bằng Python cần tiến trình, không phải luồng,
  vì GIL.
* Attention chuẩn có mật độ tính toán $d_{đầu}/2$ ở phần ma trận điểm. Chia khối với softmax trực tuyến
  cho đúng cùng kết quả mà không bao giờ tạo ma trận $S \times S$, nên byte phải giữ giảm từ bậc hai
  xuống bậc một: 67.1 MB xuống 3.1 MB mỗi đầu ở 4,096 token.
* AllReduce vòng làm mỗi GPU gửi $2(N-1)/N$ lần gradient, gần hai bản dù cụm lớn đến đâu. Với 7 tỷ tham
  số trên mạng 100 Gbps, chừng ấy byte mất 1.96 giây, gấp đôi một giây tính, nên mức sử dụng còn khoảng
  33.8 phần trăm.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Huấn luyện mô hình — khi định luật sắt trở thành công cụ hằng ngày](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch08-model-training/), dựng từ chương
[*Model Training*](https://mlsysbook.ai/vol1/training/training.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.